In [1]:
# ===== 팀 프로젝트 시작 셀 =====
import os, sys
from google.colab import userdata

TOKEN = userdata.get('GITHUB_TOKEN')
NAME = "yurim"
REPO_URL = "github.com/DA15-Lab/team3-ping-analytics.git"
PROJECT_DIR = "/content/team3-ping-analytics"

if not os.path.exists(PROJECT_DIR):
    !git clone -q https://{TOKEN}@{REPO_URL}
    %cd {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull -q

!git config user.email "{NAME}@example.com"
!git config user.name "{NAME}"

!pip install -q -r requirements.txt

sys.path.append("/content/team3-ping-analytics/src")
import preprocess

print(preprocess.module_test())
print("✅ 준비 완료")

/content/team3-ping-analytics
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 78.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 MB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 34.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 32.9 MB/s eta 0:00:00
✅ 모듈 설치 완료
✅ 준비 완료


In [ ]:
# ===== 작업 저장 셀 =====
NAME = "유림"
PROJECT_DIR = "/content/team3-ping-analytics"

%cd {PROJECT_DIR}
!git config pull.rebase false
!git config core.editor "true"
!git pull
!pip install -q -r requirements.txt

from google.colab import _message
import json

with open('notebooks/accounts_friendrequest_전처리.ipynb', 'w') as f:
    json.dump(_message.blocking_request('get_ipynb', timeout_sec=30)['ipynb'], f)

!pip freeze | grep -v "google-colab" > requirements.txt

!git add .
!git commit -m "accounts_friendrequest 전처리 완료"
!git push

/content/team3-ping-analytics
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 78.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 전처리

In [ ]:
from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
friendrequest = pd.read_csv('/content/drive/MyDrive/00_원본 csv/accounts_friendrequest.csv')

In [ ]:
friendrequest.shape

(17147175, 6)

In [ ]:
friendrequest.head()

,id,status,created_at,updated_at,receive_user_id,send_user_id
0,7,P,2023-04-17 18:29:11,2023-04-17 18:29:11,831962,837521
1,10,A,2023-04-17 18:29:11,2023-04-22 06:02:53,832151,837521
2,11,A,2023-04-17 18:29:11,2023-04-18 19:28:41,832340,837521
3,13,A,2023-04-17 18:29:11,2023-04-19 11:05:04,833041,837521
4,20,P,2023-04-17 18:29:11,2023-04-17 18:29:11,834415,837521


In [ ]:
friendrequest.dtypes

,0
id,int64
status,object
created_at,object
updated_at,object
receive_user_id,int64
send_user_id,int64


In [ ]:
friendrequest.isna().sum()

,0
id,0
status,0
created_at,0
updated_at,0
receive_user_id,0
send_user_id,0


In [ ]:
friendrequest['id'].duplicated().sum()

np.int64(0)

In [ ]:
friendrequest.duplicated().sum()

np.int64(0)

In [ ]:
friendrequest['status'].value_counts()

,count
status,
A,12878407
P,3938608
R,330160


In [ ]:
# created_at, updated_at을 문자열에서 datetime 타입으로 변환
friendrequest['created_at'] = pd.to_datetime(friendrequest['created_at'])
friendrequest['updated_at'] = pd.to_datetime(friendrequest['updated_at'])

In [ ]:
# 가장 오래된 요청 날짜 확인 (비정상 과거 날짜가 있는지 확인)
friendrequest['created_at'].min()

Timestamp('2023-04-17 18:29:11')

In [ ]:
# 가장 최근 요청 날짜 확인 (비정상 미래 날짜가 있는지 확인)
friendrequest['created_at'].max()

Timestamp('2024-05-09 09:21:47')

In [ ]:
# 본인이 본인에게 친구 요청 보낸 경우가 있는지 확인 (비정상 데이터)
(friendrequest['send_user_id'] == friendrequest['receive_user_id']).sum()

np.int64(0)

In [ ]:
import glob

found = glob.glob('/content/drive/MyDrive/**/accounts_user.csv', recursive=True)
user = pd.read_csv(found[0])

valid_ids = set(user['id'])
invalid_receive = ~friendrequest['receive_user_id'].isin(valid_ids)
invalid_send = ~friendrequest['send_user_id'].isin(valid_ids)

invalid_receive.sum()

np.int64(0)

In [ ]:
# receive_user_id, send_user_id가 accounts_user에 없는 값인지 확인
valid_ids = set(user['id'])
invalid_receive = ~friendrequest['receive_user_id'].isin(valid_ids)
invalid_send = ~friendrequest['send_user_id'].isin(valid_ids)

invalid_receive.sum()

np.int64(0)

In [ ]:
invalid_send.sum()

np.int64(0)

In [ ]:
SAVE_DIR = os.path.join(os.path.dirname(found[0]), 'processed')
os.makedirs(SAVE_DIR, exist_ok=True)

friendrequest.to_csv(os.path.join(SAVE_DIR, 'accounts_friendrequest_cleaned.csv'), index=False)